In [1]:
!git clone https://github.com/lekshmi-j/grammar-autocorrector.git

Cloning into 'grammar-autocorrector'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 86 (delta 44), reused 54 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 143.02 KiB | 5.30 MiB/s, done.
Resolving deltas: 100% (44/44), done.


In [2]:
%cd grammar-autocorrector

/content/grammar-autocorrector


In [3]:
!pip install datasets pandas


In [4]:
import pandas as pd
from datasets import load_dataset


In [5]:
dataset = load_dataset("jfleg", split="validation")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/141k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/755 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/748 [00:00<?, ? examples/s]

In [6]:
df = pd.DataFrame(dataset)


In [7]:
import spacy
nlp = spacy.load("en_core_web_sm")

def extract_pos_sequence(sentence: str) -> str:
    doc = nlp(sentence)
    return " ".join(tok.pos_ for tok in doc)


In [8]:
df["pos_seq"] = df["sentence"].astype(str).apply(extract_pos_sequence)
df["sent_len"] = df["sentence"].str.len()


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

features = ColumnTransformer(
    transformers=[
        ("word_ngrams",
         CountVectorizer(ngram_range=(1, 2), max_features=3000),
         "sentence"),

        ("pos_ngrams",
         CountVectorizer(ngram_range=(2, 3), max_features=1000),
         "pos_seq"),

        ("length",
         "passthrough",
         ["sent_len"])
    ],
    n_jobs=1   # IMPORTANT
)


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Create train_df and labels from the existing df (as per commented cells)
data = []

for i in range(len(df)):
    data.append({"sentence": df.loc[i, "sentence"], "label": 0}) # Original sentence is label 0 (ungrammatical)
    data.append({"sentence": df.loc[i, "corrections"][0], "label": 1}) # Corrected sentence is label 1 (grammatical)

train_df = pd.DataFrame(data)

# Ensure 'pos_seq' and 'sent_len' are created for train_df, as 'features' expects them
# If spacy.load("en_core_web_sm") is slow, you might want to pre-process outside this cell.
# Assuming 'extract_pos_sequence' and 'sent_len' calculation from previous cells are available
train_df["pos_seq"] = train_df["sentence"].astype(str).apply(extract_pos_sequence)
train_df["sent_len"] = train_df["sentence"].str.len()

# Split train_df into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(
    train_df,
    train_df["label"],
    test_size=0.2,
    random_state=42
)


pipeline = Pipeline([
    ("features", features),
    ("clf", LogisticRegression(max_iter=1000))
])

# Fit the pipeline using the prepared training data
pipeline.fit(X_train, y_train)


Pipeline(steps=[('features',
                 ColumnTransformer(n_jobs=1,
                                   transformers=[('word_ngrams',
                                                  CountVectorizer(max_features=3000,
                                                                  ngram_range=(1,
                                                                               2)),
                                                  'sentence'),
                                                 ('pos_ngrams',
                                                  CountVectorizer(max_features=1000,
                                                                  ngram_range=(2,
                                                                               3)),
                                                  'pos_seq'),
                                                 ('length', 'passthrough',
                                                  ['sent_len'])])),
                ('clf', LogisticRegression(max_iter=1000))])

In [12]:
import joblib
joblib.dump(pipeline, "models/grammar_detector.joblib")


['models/grammar_detector.joblib']

**Build a training DataFrame**

In [ ]:
'''import pandas as pd

data = []

for i in range(len(df)):
    data.append({"sentence": df.loc[i, "sentence"], "label": 0})
    data.append({"sentence": df.loc[i, "corrections"][0], "label": 1})

train_df = pd.DataFrame(data)
train_df.head()


**Feature engineering (MOST IMPORTANT PART)**

**Feature 1: Sentence length**

In [ ]:
'''def get_sentence_length(sentence):
    """
    Counts the number of words in a sentence.

    Parameters:
    sentence (str): A sentence from the dataset

    Returns:
    int: Number of words in the sentence
    """
    words = sentence.split()   # split sentence into words
    return len(words)          # count words


In [ ]:
'''train_df["sent_len"] = train_df["sentence"].apply(get_sentence_length)


In [ ]:
'''import nltk
nltk.download('punkt_tab')
nltk.download("averaged_perceptron_tagger_eng")




In [ ]:
'''from nltk import pos_tag
from nltk.tokenize import word_tokenize


def pos_sequence(sentence):
    return " ".join([tag for _, tag in pos_tag(word_tokenize(sentence))])

train_df["pos_seq"] = train_df["sentence"].apply(pos_sequence)

**Feature 3: Word n-grams (bag of words)**

In [ ]:
'''from sklearn.feature_extraction.text import CountVectorizer

word_vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=3000
)

**Feature 4: POS n-grams**

In [ ]:
'''pos_vectorizer = CountVectorizer(
    ngram_range=(2, 3),
    max_features=1000
)


**Combine features (feature union)**

In [ ]:
'''from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer

def extract_sentence_length(df):
    """
    Extracts sentence length as a numeric feature.

    Parameters:
    df (DataFrame): Input data containing 'sent_len' column

    Returns:
    numpy array of shape (n_samples, 1)
    """
    return df["sent_len"].values.reshape(-1, 1)
text_features = FeatureUnion([
    # 1️⃣ Word-level n-gram features (from sentence text)
    ("word_ngrams", word_vectorizer),

    # 2️⃣ POS-tag n-gram features (from POS-tagged text)
    ("pos_ngrams", pos_vectorizer),

    # 3️⃣ Sentence length feature
    ("sentence_length", FunctionTransformer(
        extract_sentence_length,
        validate=False
    ))
])



**Train ML models**

In [ ]:
'''print(train_df.columns)


In [ ]:
'''train_df["pos_text"] = train_df["pos_seq"]


In [ ]:
'''print(train_df[["sentence", "pos_text"]].head())


In [ ]:
#Train–test split
'''from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    train_df,
    train_df["label"],
    test_size=0.2,
    random_state=42
)


In [ ]:
'''word_vectorizer.fit(X_train["sentence"])


In [ ]:
'''pos_vectorizer.fit(X_train["pos_text"])


In [ ]:
'''from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import CountVectorizer
from nltk import pos_tag, word_tokenize

def pos_sequence(series):
    return series.apply(
        lambda x: " ".join(tag for _, tag in pos_tag(word_tokenize(x)))
    )

def sentence_length(series):
    return series.apply(lambda x: len(x.split())).values.reshape(-1, 1)

features = ColumnTransformer(
    transformers=[
        ("word_ngrams",
         CountVectorizer(ngram_range=(1, 2), max_features=3000),
         "sentence"),

        ("pos_ngrams",
         Pipeline([
             ("pos", FunctionTransformer(pos_sequence, validate=False)),
             ("vec", CountVectorizer(ngram_range=(2, 3), max_features=1000))
         ]),
         "sentence"),

        ("length",
         FunctionTransformer(sentence_length, validate=False),
         "sentence")
    ]
)


In [ ]:
'''from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

lr_model = Pipeline([
    ("features", features),
    ("clf", LogisticRegression(max_iter=1000))
])

lr_model.fit(train_df[["sentence"]], train_df["label"])


In [ ]:
# text_features = ColumnTransformer(
#     transformers=[
#         # Word n-grams from raw sentence text
#         ("word_ngrams", word_vectorizer, "sentence"),

#         # POS n-grams from POS sequence
#         ("pos_ngrams", pos_vectorizer, "pos_seq"),

#         # Sentence length as numeric feature
#         ("sent_len", FunctionTransformer(
#             lambda x: x.values.reshape(-1, 1),
#             validate=False
#         ), ["sent_len"])
#     ]
# )


In [ ]:
# lr_model = Pipeline([
#     ("features", text_features),
#     ("clf", LogisticRegression(max_iter=1000))
# ])


In [ ]:
'''lr_model.fit(X_train, y_train)


In [ ]:
'''print(X_train[["sentence", "pos_seq", "sent_len"]].head())


## Debugging Note: Empty Vocabulary Error

While training the baseline Logistic Regression model, I encountered the following error:

ValueError: empty vocabulary; perhaps the documents only contain stop words


Initially, this looked like a text preprocessing issue, but the dataset itself was fine. The problem turned out to be in how the feature pipeline was constructed.

---

## What Went Wrong

The model uses multiple feature types:

- Word n-grams from the `sentence` column  
- POS n-grams from the `pos_seq` column  
- Sentence length from the `sent_len` column  

I originally combined these features using `FeatureUnion`. However, `FeatureUnion` passes the entire input object to every transformer. Since the input was a pandas DataFrame, the text vectorizers did not know which column to read from. As a result, they received invalid input and produced an empty vocabulary.

The error message was misleading—the issue was not stop words, but incorrect feature wiring.

---

## How I Fixed It

I replaced `FeatureUnion` with `ColumnTransformer`. This allows each transformer to explicitly specify which DataFrame column it should operate on.

```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

text_features = ColumnTransformer(
    transformers=[
        ("word_ngrams", word_vectorizer, "sentence"),
        ("pos_ngrams", pos_vectorizer, "pos_seq"),
        ("sent_len", FunctionTransformer(
            lambda x: x.values.reshape(-1, 1),
            validate=False
        ), ["sent_len"])
    ]
)


FeatureUnion should be used only when all transformers operate on the same input.

When working with DataFrames and multiple feature columns, ColumnTransformer is the correct choice.

An “empty vocabulary” error often points to incorrect input being passed to a vectorizer, not an issue with the text itself.

In [ ]:
'''from sklearn.metrics import classification_report

y_pred = lr_model.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
'''from sklearn.naive_bayes import MultinomialNB

nb_model = Pipeline([
    ("features", text_features),
    ("clf", MultinomialNB())
])

nb_model.fit(X_train, y_train)


In [ ]:
'''mkdir models


In [ ]:
'''import os
os.makedirs("models", exist_ok=True)


In [ ]:
'''lr_model.predict(X_test[:5])


In [ ]:
'''import joblib

joblib.dump(lr_model, "models/grammar_detector.joblib")


In [ ]:
'''import os
os.path.exists("models/grammar_detector.joblib")


In [ ]:
'''!pip install pyspellchecker


In [ ]:
'''from src.corrector import correct_sentence

print(correct_sentence("He go to market"))
print(correct_sentence("She went to school"))
